# 03 — Feature Selection (Train‑only, Leakage‑safe)

This notebook restores the canonical configuration and splits, builds train‑only frames, and performs categorical folding, mutual information scoring, and top‑K selection.


## Section 0.05 — Restore Canonical Config (Single Source)

**What this does**
- Restores `RANDOM_STATE`, `DATA_PATH`, `TARGET_COL`, `STAGE_ROOT`, `OUT_ROOT` from `CANON.json` persisted by 01.

**Must**
- Run first in isolated kernels to keep a single source of truth.

**Outputs**
- In‑memory config variables identical to 01 (no redefinition).


In [ ]:
# ======================================================
# Section 0.05 — Restore Canonical Config (Single Source)
#   • Loads CANON.json persisted by 01 into globals
#   • Supports isolated kernel execution
# ======================================================
print(">>> Section 0.05 — Restore Canonical Config (Single Source)")

import json
from pathlib import Path

candidates = [Path("CANON.json"), Path("staging") / "CANON.json"]
src = next((p for p in candidates if p.exists()), None)
assert src is not None, "[0.05] CANON.json not found. Run 01-Setup_Preflight (0.1) first."

cfg = json.loads(src.read_text(encoding="utf-8"))
from pathlib import Path as _Path
RANDOM_STATE = cfg["RANDOM_STATE"]
DATA_PATH    = _Path(cfg["DATA_PATH"])
TARGET_COL   = cfg["TARGET_COL"]
STAGE_ROOT   = _Path(cfg["STAGE_ROOT"])
OUT_ROOT     = _Path(cfg["OUT_ROOT"])

print(f"[0.05] Loaded CANON from {src}")
print(f"[0.05] STAGE_ROOT={STAGE_ROOT.resolve()} | OUT_ROOT={OUT_ROOT.resolve()}")


## Section 0.6 — Restore Staging Training Data (Auto‑Restore)

**What this does**
- Restores `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, `y_test` from `STAGE_ROOT/splits`.

**Must**
- Run before any section that expects splits in memory.


In [ ]:
# ======================================================
# Section 0.6 — Restore Staging Training Data (Auto‑Restore)
# ======================================================
print(">>> Section 0.6 — Restore Staging Training Data (Auto‑Restore)")

from pathlib import Path
import joblib

assert 'STAGE_ROOT' in globals(), "[0.6] STAGE_ROOT not defined. Run 0.05 first."
splits_dir = Path(STAGE_ROOT) / "splits"
assert splits_dir.exists(), f"[0.6] Missing {splits_dir}. Run 01-Setup_Preflight (0.6 Save)."

def _load(name):
    p = splits_dir / f"{name}.joblib"
    assert p.exists(), f"[0.6] Missing split artefact: {p}"
    return joblib.load(p)

X_train = _load("X_train"); y_train = _load("y_train")
X_val   = _load("X_val");   y_val   = _load("y_val")
X_test  = _load("X_test");  y_test  = _load("y_test")

print(f"[0.6] Restored: X_train={getattr(X_train,'shape',None)}, X_val={getattr(X_val,'shape',None)}, X_test={getattr(X_test,'shape',None)}")
print("[0.6] Auto-restore complete.")


## Section 3.0 — Build training `df` and `X` (no leakage)

**What this does**
- Constructs training `df` by concatenating `X_train` with `y_train` (renamed to `TARGET_COL`).
- Builds feature matrix `X` by excluding forbidden columns: `{ 'payload','attack_cat','label_str', TARGET_COL }`.

**Must**
- Use train split only to avoid target leakage.
- Keep `X` and `df` in memory for Sections 3.1–3.4.


In [ ]:
# ======================================================
# Section 3.0 — Build training df and X (no leakage)
#   • Satisfies 3.1/3.3 preconditions: df has TARGET_COL; X has only allowed features
# ======================================================
print(">>> Section 3.0 — Build training df and X (no leakage)")

import pandas as pd
import numpy as np

for v in ("X_train","y_train","TARGET_COL"):
    assert v in globals(), f"[3.0] Missing {v}; run 0.6/0.05 first."

# Params (single source for this notebook)
MIN_CAT_FREQ = 5   # count threshold; must be ≥ 1
TOPK_MI      = 200 # keep top‑K features by MI

# Normalise y_train to a Series named TARGET_COL with aligned index
if hasattr(y_train, "values") and hasattr(y_train, "index"):
    y_series = y_train if y_train.name == TARGET_COL else pd.Series(y_train.values, index=X_train.index, name=TARGET_COL)
elif isinstance(y_train, np.ndarray):
    assert len(y_train) == len(X_train), "[3.0] y_train length mismatch"
    y_series = pd.Series(y_train, index=X_train.index, name=TARGET_COL)
else:
    y_series = pd.Series(np.asarray(y_train), index=X_train.index, name=TARGET_COL)

df = pd.concat([X_train.copy(), y_series], axis=1)

forbidden = {'payload', 'attack_cat', 'label_str', TARGET_COL}
X = df.drop(columns=[c for c in forbidden if c in df.columns], errors='ignore').copy()

print(f"[3.0] df shape={df.shape}; X shape={X.shape}; target={TARGET_COL!r}")
print(f"[3.0] Dropped forbidden columns present: {[c for c in forbidden if c in df.columns]}")
print("[3.0] Train‑only frame ready for 3.1 and 3.3.")


## Section 3.1 — Categorical audit + rare‑value folding (train‑based; applied to val/test)

**What this does**
- Identifies high‑cardinality categoricals on **train** and folds categories with frequency `< MIN_CAT_FREQ` to `__other__`.
- Applies the same folding to `X_val` and `X_test` to maintain schema consistency.

**Must**
- Never use validation/test to decide rare categories (no leakage).


In [ ]:
# ======================================================
# Section 3.1 — Categorical audit + rare‑value folding (train‑based; applied to val/test)
# ======================================================
print(">>> Section 3.1: start")

import pandas as pd

assert all(v in globals() for v in ["X_train","X_val","X_test"]), "[3.1] run 0.6 first (splits)"
assert "MIN_CAT_FREQ" in globals(), "[3.1] define MIN_CAT_FREQ in Section 3.0"
assert isinstance(MIN_CAT_FREQ, (int, float)) and MIN_CAT_FREQ >= 1, "[3.1] MIN_CAT_FREQ must be ≥ 1"

exclude = {"payload", "attack_cat", "label_str"}
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" and c not in exclude]

TOPN_REPORT = 10
card_train = {c: int(X_train[c].nunique(dropna=False)) for c in cat_cols}
topn_before = sorted(card_train.items(), key=lambda kv: kv[1], reverse=True)[:TOPN_REPORT]
print("[3.1] top categorical columns by cardinality (train, before folding):")
for c, k in topn_before:
    print(f"    {c:>24s} → {k:>6d} unique")

rare_map = {}
for c in cat_cols:
    vc = X_train[c].value_counts(dropna=False)
    rare_set = set(vc[vc < MIN_CAT_FREQ].index.tolist())
    if rare_set:
        rare_map[c] = rare_set

def _fold_rare(series: pd.Series, rare_set: set) -> pd.Series:
    mask = series.isin(rare_set) & series.notna()
    out = series.copy()
    out.loc[mask] = "__other__"
    return out

for c, rset in rare_map.items():
    X_train[c] = _fold_rare(X_train[c], rset)
    if c in X_val.columns:
        X_val[c] = _fold_rare(X_val[c], rset)
    if c in X_test.columns:
        X_test[c] = _fold_rare(X_test[c], rset)

card_train_after = {c: int(X_train[c].nunique(dropna=False)) for c in cat_cols}
topn_after = sorted(card_train_after.items(), key=lambda kv: kv[1], reverse=True)[:TOPN_REPORT]
print("[3.1] top categorical columns by cardinality (train, after folding):")
for c, k in topn_after:
    print(f"    {c:>24s} → {k:>6d} unique")

print(f"[3.1] folded rare categories (< {MIN_CAT_FREQ}) into '__other__' for {len(rare_map)} columns.")


## Section 3.3 — Mutual Information (MI) Scoring (Canonical)

**What this does**
- Computes MI between each column of `X` (train‑only) and `TARGET_COL`.
- Persists sorted MI scores for downstream selection.

**Must**
- Ensure `df[TARGET_COL]` is binary encoded in `{0,1}` before MI.
- No validation/test data in MI computation.


In [ ]:
# ======================================================
# Section 3.3 — Mutual Information (MI) Scoring (Canonical)
#   • Computes MI for each column in X vs TARGET_COL
#   • Produces sorted mi_series and persists to staging
# ======================================================
print(">>> Section 3.3 — Mutual Information (MI) Scoring (Canonical)")

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_selection import mutual_info_classif

assert 'X' in globals(), "[3.3] X must be available from Section 3.0"
assert 'df' in globals(), "[3.3] df must be available from Section 3.0"
assert TARGET_COL in df.columns, f"[3.3] Missing target column: {TARGET_COL!r}"

y = df[TARGET_COL].astype(int).values
assert set(np.unique(y)).issubset({0, 1}), "[3.3] Target must be binary in {0,1}"

X_work = X.copy()
cat_cols_local = [c for c in X_work.columns if str(X_work[c].dtype) in ("object", "category")]
num_cols_local = [c for c in X_work.columns if c not in cat_cols_local]

for c in cat_cols_local:
    X_work[c] = pd.factorize(X_work[c].astype("category"), sort=True)[0]

if num_cols_local:
    X_work[num_cols_local] = X_work[num_cols_local].fillna(X_work[num_cols_local].median())
if cat_cols_local:
    X_work[cat_cols_local] = X_work[cat_cols_local].fillna(-1)

discrete_mask = np.array([col in cat_cols_local for col in X_work.columns], dtype=bool)

mi = mutual_info_classif(
    X_work.values,
    y,
    discrete_features=discrete_mask,
    random_state=RANDOM_STATE
)
mi_series = pd.Series(mi, index=X_work.columns, name="MI").sort_values(ascending=False)

feat_dir = Path(STAGE_ROOT) / "feat"
feat_dir.mkdir(parents=True, exist_ok=True)
mi_path = feat_dir / "mi_series.csv"
mi_series.to_csv(mi_path, header=True)

print(f"[3.3] Computed MI for {len(mi_series)} features; top 10:")
print(mi_series.head(10).to_string())
print(f"[3.3] Saved MI series → {mi_path}")
print("[3.3] MI scoring complete.")


## Section 3.4 — Feature manifest + top‑K MI selection

**What this does**
- Selects top‑`K` features by MI and builds a feature manifest for reproducibility.
- Persists `feature_manifest.json` and (optionally) a `joblib` of the selected training matrix.

**Must**
- Use `TOPK_MI` from Section 3.0.
- Persist artefacts under `STAGE_ROOT/feat`.


In [ ]:
# =====================================================
# Section 3.4 — Feature manifest + top‑K MI selection
# =====================================================
print(">>> Section 3.4: start")

import json, joblib
from pathlib import Path

assert 'mi_series' in globals(), "[3.4] run 3.3 first to compute MI scores"
assert 'X' in globals(), "[3.4] X must be available from 3.0"
assert 'TOPK_MI' in globals(), "[3.4] TOPK_MI must be set in 3.0"

keep_names = mi_series.head(TOPK_MI).index.tolist()

X_feat = X[keep_names].copy()
feat_dir = Path(STAGE_ROOT) / "feat"
feat_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "base_columns": X.columns.tolist(),
    "cat_cols": [c for c in X_feat.columns if X_feat[c].dtype == "object"],
    "num_cols": [c for c in X_feat.columns if c not in X_feat.select_dtypes(include=["object"]).columns],
    "mi_selected": keep_names,
    "params": {
        "MIN_CAT_FREQ": int(MIN_CAT_FREQ),
        "TOPK_MI": int(TOPK_MI)
    },
    "shapes": {
        "X_feat": list(X_feat.shape)
    }
}

(feat_dir / "feature_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
joblib.dump(X_feat, feat_dir / "X_feat.joblib")

print(f"[3.4] kept {len(keep_names)} features → {feat_dir}")
X_feat.head()
